# EXP-33: BERTimbau + EXP-24 Pipeline Blend + Binary Detectors

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-24 (0.81021 LB)  
**Strategy:** Adicionar BERTimbau fine-tuned como 8º modelo L1 no pipeline EXP-24 + detector binário classe 0 vs 2  

**Insight:** Transformers sozinhos (EXP-10: 0.62) são fracos. Mas como membro de ensemble com TF-IDF, podem capturar padrões semânticos que TF-IDF perde. O truque é usar BERT como feature extractor, não como modelo final.

**Novidades vs EXP-24:**
1. **BERTimbau fine-tuned** (3 epochs, lr=2e-5) → probabilities como 8º input do meta-learner
2. **Binary class-0 detector**: SVC focado em {0,2} para corrigir confusão 0↔2 (ideia do notebook externo)
3. **Blend alpha search**: BERT_weight testado de 0.0 a 0.3 no ensemble

**Runtime:** GPU T4 (BERTimbau) + CPU (TF-IDF pipeline)

In [ ]:
import os, re, hashlib, warnings, time, gc
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Train: {train.shape}, Test: {test.shape}, Device: {DEVICE}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing (IDENTICO ao EXP-24)
# =============================================================================

def clean_achados(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_conclusion(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    if match:
        c = match.group(1).strip()
        c = re.sub(r'[0-9]+,[0-9]+', 'NUM', c)
        c = re.sub(r'[0-9]+', 'NUM', c)
        return c
    return ""

def stable_hash(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()

train["achados"]    = train["report"].apply(clean_achados)
train["full"]       = train["report"].apply(clean_full)
train["conclusion"] = train["report"].apply(extract_conclusion)
test["achados"]     = test["report"].apply(clean_achados)
test["full"]        = test["report"].apply(clean_full)
test["conclusion"]  = test["report"].apply(extract_conclusion)
train["group"]      = train["report"].apply(stable_hash)

y = train["target"].astype(int).values
groups = train["group"].values
print(f'Preprocessing done.')

In [ ]:
# =============================================================================
# Cell 3: BERTimbau Fine-Tuning with GroupKFold OOF

# Auto-detect model: find ANY config.json under /kaggle/input/
import subprocess as _sp
MODEL_NAME = None

if os.path.exists('/kaggle/input'):
    try:
        result = _sp.run(
            ['find', '/kaggle/input', '-name', 'config.json', '-type', 'f'],
            capture_output=True, text=True, timeout=10
        )
        config_files = [l.strip() for l in result.stdout.strip().split('\n') if l.strip()]
        print(f"Found config.json files: {config_files}")
        
        for cfg_path in config_files:
            cfg_dir = os.path.dirname(cfg_path)
            dir_files = os.listdir(cfg_dir)
            print(f"  {cfg_dir}: {dir_files}")
            # Accept any dir with config.json + model weights or tokenizer
            has_model = any(f.endswith(('.bin', '.safetensors')) for f in dir_files)
            has_tokenizer = any('vocab' in f.lower() or 'tokenizer' in f.lower() for f in dir_files)
            if has_model or has_tokenizer:
                MODEL_NAME = cfg_dir
                print(f"Using local model: {MODEL_NAME}")
                break
    except Exception as e:
        print(f"Error scanning: {e}")

if MODEL_NAME is None:
    MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
    print(f"WARNING: No local model found! Falling back to: {MODEL_NAME}")

MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

class ReportDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(train['report'].values, y, groups))

# Compute class weights for focal loss
class_counts = np.bincount(y, minlength=N_CLASSES)
class_weights = torch.tensor(
    (1.0 / class_counts) * len(y) / N_CLASSES, dtype=torch.float32
).to(DEVICE)

bert_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
bert_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)

t0 = time.time()
for fi, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n=== BERT Fold {fi} ===")
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES
    ).to(DEVICE)
    
    train_ds = ReportDataset(train['report'].values[tr_idx], y[tr_idx], tokenizer, MAX_LEN)
    val_ds = ReportDataset(train['report'].values[va_idx], y[va_idx], tokenizer, MAX_LEN)
    test_ds = ReportDataset(test['report'].values, tokenizer=tokenizer, max_len=MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Training
    best_val_f1 = 0
    best_state = None
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_preds = []
        val_probs = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
                val_probs.append(probs)
                val_preds.extend(probs.argmax(axis=1))
        
        val_f1 = f1_score(y[va_idx], val_preds, average='macro')
        print(f"  Epoch {epoch}: loss={train_loss/len(train_loader):.4f}, val_f1={val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Load best model for this fold
    model.load_state_dict(best_state)
    model.eval()
    
    # OOF predictions
    val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            val_probs.append(probs)
    bert_oof[va_idx] = np.vstack(val_probs)
    
    # Test predictions
    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            test_probs.append(probs)
    bert_test_acc += np.vstack(test_probs) / N_FOLDS
    
    print(f"  Fold {fi} best val F1: {best_val_f1:.4f}")
    
    del model, best_state
    gc.collect()
    torch.cuda.empty_cache()

bert_oof_f1 = f1_score(y, np.argmax(bert_oof, axis=1), average='macro')
print(f"\nBERT OOF F1: {bert_oof_f1:.4f}")
print(f"BERT done in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 4: Dense Features + TF-IDF (IDENTICO ao EXP-24)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    feat['report_length']   = text_col.apply(len)
    feat['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']  = text_col.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']      = text_col.str.contains(r'biopsy|bi[oó]psia|resultado de cine|carcinoma', regex=True).astype(int)
    feat['word_count']        = text_col.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text_col.str.count('\n') + 1
    feat['achados_length']    = df['achados'].str.len() if 'achados' in df.columns else 0
    feat['conclusion_length'] = df['conclusion'].str.len() if 'conclusion' in df.columns else 0
    feat['has_nodule']        = text_col.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification'] = text_col.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']     = text_col.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']     = text_col.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']     = text_col.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']    = text_col.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']        = text_col.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']    = text_col.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']  = text_col.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']   = text_col.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']     = text_col.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']     = text_col.str.contains(r'bilateral', regex=True).astype(int)
    feat['has_birads_explicit'] = text_col.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text_col.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2 - feat['has_gross_calc'] * 2
    )
    return csr_matrix(feat.values.astype(np.float32))

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)

print("Building TF-IDF...")
tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

tfidf_C = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_C = tfidf_C.fit_transform(train["conclusion"].values)
X_test_C  = tfidf_C.transform(test["conclusion"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()
X_train_full_dense = hstack([X_train_F, X_train_C, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_C, X_test_dense]).tocsr()

print("Features built.")

In [ ]:
# =============================================================================
# Cell 5: EXP-24 Level 1 (7 models) — IDENTICO
# =============================================================================

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

L1_MODELS = [
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     X_train_full_dense, X_test_full_dense),
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     X_train_F, X_test_F),
]

oof_probas = {}
test_probas = {}
t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  {name}: OOF F1={oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

# Add BERT as 8th model
oof_probas['bert'] = bert_oof
test_probas['bert'] = bert_test_acc
print(f"  bert: OOF F1={bert_oof_f1:.4f}")
print(f"\n8 models done in {time.time()-t0:.0f}s (CPU part)")

In [ ]:
# =============================================================================
# Cell 6: Binary Class-0 Detector (ideia do notebook externo)
# =============================================================================

print("Training binary class-0 detector on {0, 2} subset...")

mask_02 = np.isin(y, [0, 2])
y_02 = (y[mask_02] == 0).astype(int)  # 1 = class 0, 0 = class 2

bin0_svc = CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', random_state=42, max_iter=5000),
    cv=3, method='sigmoid'
)
bin0_svc.fit(X_train_F[mask_02], y_02)
bin0_test_proba = bin0_svc.predict_proba(X_test_F)[:, 1]  # P(class 0)

# Also get OOF for the binary detector to find optimal threshold
bin0_oof = np.zeros(mask_02.sum(), dtype=np.float64)
gkf_02 = GroupKFold(n_splits=3)
for fi, (tr_idx, va_idx) in enumerate(gkf_02.split(X_train_F[mask_02], y_02, groups[mask_02])):
    m = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', random_state=42, max_iter=5000),
        cv=3, method='sigmoid'
    )
    m.fit(X_train_F[mask_02][tr_idx], y_02[tr_idx])
    bin0_oof[va_idx] = m.predict_proba(X_train_F[mask_02][va_idx])[:, 1]

# Find best threshold
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_02, bin0_oof)
print(f"Binary detector AUC: {auc:.4f}")

best_bin_thr = 0.50
best_bin_f1 = 0
for thr in np.arange(0.30, 0.75, 0.01):
    preds_02 = (bin0_oof > thr).astype(int)
    f = f1_score(y_02, preds_02, average='macro')
    if f > best_bin_f1:
        best_bin_f1 = f
        best_bin_thr = thr

print(f"Best binary threshold: {best_bin_thr:.2f} (F1={best_bin_f1:.4f})")

In [ ]:
# =============================================================================
# Cell 7: Level 2 — Meta-Learner with 8 models (7 CPU + BERT)
# =============================================================================

model_names = list(oof_probas.keys())
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])
print(f"Meta-features: {X_meta_train.shape[1]} ({len(model_names)} models x {N_CLASSES} classes)")

meta_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    meta_lr = LogisticRegression(class_weight='balanced', C=1.0, max_iter=5000,
                                 solver='lbfgs', multi_class='multinomial', random_state=42)
    meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
    meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS
    fold_f1 = f1_score(y[va_idx], np.argmax(meta_oof[va_idx], axis=1), average='macro')
    print(f"Meta Fold {fi}: F1={fold_f1:.4f}")

meta_f1 = f1_score(y, np.argmax(meta_oof, axis=1), average='macro')
print(f"\nMeta-learner (8 models) OOF: {meta_f1:.4f}")

# V12 blend (same as EXP-24, without BERT)
oof_svc_ens = 0.25 * oof_probas['svc_A'] + 0.40 * oof_probas['svc_F'] + 0.35 * oof_probas['svc_F2']
oof_v12 = 0.70 * oof_svc_ens + 0.30 * oof_probas['lgb1']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style OOF: {v12_f1:.4f}")

# Hybrid
oof_hybrid = 0.5 * meta_oof + 0.5 * oof_v12
hybrid_f1 = f1_score(y, np.argmax(oof_hybrid, axis=1), average='macro')
print(f"Hybrid OOF: {hybrid_f1:.4f}")

# Select best
results = {'meta': (meta_f1, meta_oof, meta_test_acc)}
te_svc_ens = 0.25 * test_probas['svc_A'] + 0.40 * test_probas['svc_F'] + 0.35 * test_probas['svc_F2']
te_v12 = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']
results['v12'] = (v12_f1, oof_v12, te_v12)
results['hybrid'] = (hybrid_f1, oof_hybrid, 0.5 * meta_test_acc + 0.5 * te_v12)

best = max(results, key=lambda k: results[k][0])
final_score, oof_blend, te_blend = results[best]
print(f"\nBest: {best} (F1={final_score:.4f})")

In [ ]:
# =============================================================================
# Cell 8: Threshold Search + Binary Detector + Guardrails + Submission
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

baseline_f1 = f1_score(y, np.argmax(oof_blend, axis=1), average='macro')
print(f'Baseline OOF: {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: threshold={best_cls_thr:.2f}, F1={best_cls_f1:.4f}')

print(f'\nOOF with thresholds: {current_f1:.4f}')
print(classification_report(y, apply_thresholds(oof_blend, best_thresholds), digits=4))

# Apply to test
test_preds = apply_thresholds(te_blend, best_thresholds)

# Binary class-0 override
n_overrides = ((bin0_test_proba > best_bin_thr) & (test_preds == 2)).sum()
test_preds[(bin0_test_proba > best_bin_thr) & (test_preds == 2)] = 0
print(f"\nBinary class-0 overrides: {n_overrides}")

test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            return mentioned
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b|neoplasia maligna', text):
        return 6
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4:
        return 5
    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text):
            return 2
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('\nPrediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'Submission saved! Shape: {submission.shape}')